# पाठ ०२ - माइक्रोसफ्ट एजेन्ट फ्रेमवर्क अन्वेषण

**माइक्रोसफ्ट एजेन्ट फ्रेमवर्क (MAF)** एकीकृत फ्रेमवर्क हो जसले AI एजेन्टहरू निर्माण गर्न सहयोग गर्छ। यसले चार मुख्य निर्माण बलकहरू सहित एक सफा, संरचित आर्किटेक्चर प्रदान गर्दछ:

- **Client** – AI मोडेल अन्त्य बिन्दुमा जडान गर्दछ र संवाद व्यवस्थापन गर्छ
- **Agent** – निर्देशनहरू र उपकरण परिभाषाहरूको साथ क्लाइन्टलाई आवरण गर्दछ
- **Tools** – मोडेलले कल गर्न सक्ने कस्टम कार्यहरूसँग एजेन्ट क्षमताहरू विस्तार गर्दछ
- **Session** – बहु-चरण अन्तरक्रियाहरूको लागि संवाद इतिहास कायम राख्छ

यस पाठमा, हामी यी अवधारणाहरू प्रयोग गरी गन्तव्य उपलब्धता जाँच गर्ने **यात्रा बुकिंग एजेन्ट** निर्माण गर्नेछौं।


## सेटअप


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## एजेन्ट फ्रेमवर्क आर्किटेक्चर बुझ्न

Microsoft एजेन्ट फ्रेमवर्कले तहयुक्त आर्किटेक्चर अनुसरण गर्छ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ग्राहक** – एउटा `AzureAIProjectAgentProvider` Azure OpenAI डिप्लोइमेण्टसँग जडान हुन्छ। यसले प्रमाणिकरण, अनुरोध ढाँचा बनाउने, र प्रतिक्रिया पार्सिङ्ग ह्यान्डल गर्छ।
2. **एजेन्ट** – ग्राहकबाट `provider.create_agent()` मार्फत सिर्जना गरिन्छ, एजेन्टले मोडेल पहुँचलाई निर्देशनहरू (सिस्टम प्रॉम्प्ट) र उपकरणहरूसँग संयोजन गर्छ।
3. **उपकरणहरू** – `@tool` द्वारा सजिएका Python फङ्सनहरू जुन एजेन्टले क्रियाकलाप गर्न वा डेटा पुनःप्राप्त गर्न बोलाउन सक्छ।
4. **सेसन** – एउटा `AgentSession` वस्तु (एजेन्टबाट `agent.create_session()` मार्फत सिर्जना गरिएको) जुन संवाद इतिहास संग्रह गर्छ, जसले बहु-चरण संवाद सक्षम पार्छ जहाँ एजेन्टले पहिलेको सन्दर्भ याद गर्छ।

हामी प्रत्येक तहलाई चरणबद्ध रूपमा बनाऔं।


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटरसँग उपकरणहरू थप्दै

उपकरणहरूले एजेन्टहरूलाई पाठ उत्पादन गरेर मात्र होइन, कार्यहरू लिन अनुमति दिन्छन्। `@tool` डेकोरेटरले सामान्य Python फंक्शनलाई एजेण्टले कल गर्न सकिने वस्तुमा रूपान्तरण गर्छ।

मुख्य बुँदाहरू:
- मोडेलले प्रत्येक प्यारामिटर बुझ्न `Annotated[type, "description"]` प्रयोग गर्नुहोस्।
- डकस्ट्रिङले उपकरणको विवरणको रूपमा काम गर्छ जुन मोडेलले देख्छ।
- `approval_mode="never_require"` को अर्थ उपकरण स्वतः प्रयोगकर्ताको स्वीकृति बिना चल्छ।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## उपकरणहरू सहित एजेन्ट बनाउँदै

अब हामी क्लाइंट, निर्देशनहरू, र उपकरणहरूलाई एउटा एजेन्टमा संयोजन गर्छौं। `निर्देशनहरू` प्रणाली प्रॉम्प्टको रूपमा काम गर्छन् — तिनीहरूले एजेन्टको व्यक्तित्व र व्यवहार परिभाषित गर्छन्।


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## सत्रहरूसँग बहु-पटक कुराकानीहरू

एउटा `AgentSession` (`agent.create_session()` मार्फत सिर्जना गरिएको) ले कुराकानीमा भएका सबै सन्देशहरूलाई ट्रयाक गर्छ। एउटै सत्रलाई प्रत्येक `agent.run()` कलमा पास गर्दा, एजेन्टसँग पुरै कुराकानी इतिहास पहुँच हुन्छ र यो अघिल्ला सन्देशहरूलाई सम्झन सक्छ।

हामी `tools=[check_destination_availability]` पास गरिरहेका छौं ताकि एजेन्टले प्रत्येक पटक हाम्रो उपलब्धता जाँचकर्तालाई कल गर्न सकोस्।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

यस पाठमा तपाईले माइक्रोसफ्ट एजेन्ट फ्रेमवर्कका चार स्तम्भहरू अन्वेषण गर्नुभयो:

| Concept | तपाईले के सिक्नुभयो |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` क्रेडेन्शियल-आधारित प्रमाणीकरण सहित Azure OpenAI सँग जडान हुन्छ |
| **Agent** | `provider.create_agent()` मोडेल जडानलाई निर्देशनहरू र नाम सहित बाँध्दछ |
| **Tools** | `@tool` डेकोरेटरले एजेन्टले कल गर्ने Python कार्यहरूलाई अनावरण गर्दछ |
| **Session** | `agent.create_session()` धेरै पटकको भाषण इतिहास कायम गर्दछ |

यी निर्माण ब्लकहरूले सँगै प्राकृतिक संवादहरू राख्न, बाह्य कार्यहरू कल गर्न, र सन्दर्भ कायम राख्न सक्ने एजेन्टहरू सिर्जना गर्छन् — आगामी पाठहरूमा थप उन्नत एजेन्टिक ढाँचाहरूको आधार।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो कागजात AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताको लागि प्रयास गर्छौं, तर कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटि वा अस्पष्टता हुन सक्छ। मूल कागजात यसको मूल भाषामा अधिकारिक स्रोत मानिनु पर्छ। महत्वपूर्ण जानकारीको लागि पेशेवर मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट आउने कुनै पनि गलत बुझाइ वा भ्रमको जिम्मेवारी हामी लिन सक्दैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
